In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

torch.manual_seed(1337)

In [10]:
lstm = nn.LSTM(3, 3)

In [11]:
inputs = [torch.randn(1, 3) for _ in range(5)]
inputs

[tensor([[1.0517, 0.2501, 1.5759]]),
 tensor([[0.6495, 0.9631, 1.0905]]),
 tensor([[-0.4931, -0.9682, -0.1791]]),
 tensor([[ 0.2050, -0.7801, -1.0036]]),
 tensor([[ 0.0850,  1.0277, -0.3999]])]

In [12]:
hidden = (torch.randn(1, 1, 3),
          torch.randn(1, 1, 3))
for i in inputs:
    print(i.view(1, 1, -1))
    out, hidden = lstm(i.view(1, 1, -1), hidden)
    print(out)

tensor([[[1.0517, 0.2501, 1.5759]]])
tensor([[[ 0.5414, -0.1203,  0.3276]]], grad_fn=<MkldnnRnnLayerBackward0>)
tensor([[[0.6495, 0.9631, 1.0905]]])
tensor([[[0.1442, 0.0049, 0.2731]]], grad_fn=<MkldnnRnnLayerBackward0>)
tensor([[[-0.4931, -0.9682, -0.1791]]])
tensor([[[-0.1083, -0.0133,  0.1931]]], grad_fn=<MkldnnRnnLayerBackward0>)
tensor([[[ 0.2050, -0.7801, -1.0036]]])
tensor([[[-0.1856, -0.2745,  0.0262]]], grad_fn=<MkldnnRnnLayerBackward0>)
tensor([[[ 0.0850,  1.0277, -0.3999]]])
tensor([[[-0.2300,  0.1136,  0.1939]]], grad_fn=<MkldnnRnnLayerBackward0>)


In [13]:
def prepare_sequence(seq, to_ix):
    idxs = [to_ix[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)


training_data = [
    # Tags are: DET - determiner; NN - noun; V - verb
    # For example, the word "The" is a determiner
    ("The dog ate the apple".split(), ["DET", "NN", "V", "DET", "NN"]),
    ("Everybody read that book".split(), ["NN", "V", "DET", "NN"])
]
word_to_ix = {}
# For each words-list (sentence) and tags-list in each tuple of training_data
for sent, tags in training_data:
    for word in sent:
        if word not in word_to_ix:  # word has not been assigned an index yet
            word_to_ix[word] = len(word_to_ix)  # Assign each word with a unique index
print(word_to_ix)
tag_to_ix = {"DET": 0, "NN": 1, "V": 2}  # Assign each tag with a unique index

EMBEDDING_DIM = 6
HIDDEN_DIM = 6
OUTPUT_DIM = len(tag_to_ix) # One for each tag type

{'The': 0, 'dog': 1, 'ate': 2, 'the': 3, 'apple': 4, 'Everybody': 5, 'read': 6, 'that': 7, 'book': 8}


In [14]:
class LSTMTagger(nn.Module):
    def __init__(self, vocab_size: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBEDDING_DIM)
        self.lstm = nn.LSTM(EMBEDDING_DIM, HIDDEN_DIM)
        self.head = nn.Linear(HIDDEN_DIM, OUTPUT_DIM)

    def forward(self, x): # x is a sentence, str[]
        sentence_embedding = self.embedding(x) # (len(x), EMBEDDING_DIM)
        output, _ = self.lstm(sentence_embedding.view(len(x), 1, -1)) # (len(x), 1, HIDDEN_DIM)
        logits = self.head(output.view(len(x), -1)) # (len(x), OUTPUT_DIM)
        tag_scores = F.log_softmax(logits, dim=1) # (len(x), OUTPUT_DIM)
        return tag_scores

In [22]:
model = LSTMTagger(len(word_to_ix))
loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)
    print(tag_scores)
    print(torch.argmax(tag_scores, dim=1))

tensor([[-1.0467, -1.1937, -1.0619],
        [-0.9623, -1.3054, -1.0586],
        [-0.9884, -1.3472, -1.0000],
        [-1.0082, -1.3114, -1.0060],
        [-0.9837, -1.2941, -1.0443]])
tensor([0, 0, 0, 2, 0])


In [23]:
EPOCHS = 300

for epoch in range(EPOCHS):
    for sentence, tags in training_data:
        model.zero_grad()

        prepared_sentence = prepare_sequence(sentence, word_to_ix)
        prepared_tags = prepare_sequence(tags, tag_to_ix)
        tag_scores = model(prepared_sentence)
        loss = loss_function(tag_scores, prepared_tags)

        loss.backward()
        optimizer.step()

In [25]:
with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)

    # The sentence is "the dog ate the apple".  i,j corresponds to score for tag j
    # for word i. The predicted tag is the maximum scoring tag.
    # Here, we can see the predicted sequence below is 0 1 2 0 1
    # since 0 is index of the maximum value of row 1,
    # 1 is the index of maximum value of row 2, etc.
    # Which is DET NOUN VERB DET NOUN, the correct sequence!
    print(torch.argmax(tag_scores, dim=1))

tensor([0, 1, 2, 0, 1])
